# Layer 3 — Fitur Konteks Struk

Tidak ada identitas pelanggan, jadi satu baris adalah satu pasangan **struk × aturan**.
Baris dibuat hanya jika antecedent sudah ada di struk. Label `y = 1` bila consequent juga ada di struk yang sama.
Itulah keputusan “apakah consequent berpotensi ditambahkan ke keranjang yang sedang aktif”.

Delapan fitur:

1. `current_basket_size`, jumlah kode barang berbeda di struk.
2. `hour_of_day`, jam dari kolom `JAM`.
3. `day_of_week`, Senin = 0 sampai Minggu = 6.
4. `is_weekend`, 1 untuk Sabtu dan Minggu.
5. `rule_confidence` dan `rule_lift`, dari Apriori Januari–Februari.
6. `antecedent_rate`, pangsa struk Januari–Februari yang memuat antecedent. Maret tidak masuk perhitungan ini.
7. `interest_lift = antecedent_rate × rule_lift`. Ini menggantikan skor loyalitas member: seberapa kuat aturan itu relatif terhadap seberapa lazim antecedent di toko.

`interest_confidence = antecedent_rate × rule_confidence` tidak dijadikan kolom tersendiri karena hanya mengalikan dua fitur yang sudah ada. `interest_lift` yang dipertahankan, karena lift sudah menormalkan kebetulan.

Data latih adalah struk Januari–Februari. Data uji adalah struk Maret. Aturan tidak ditambang ulang dari Maret.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

def find_project_dir() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "outputs_ril" / "apriori_rules_ril.csv").exists():
            return candidate
    raise FileNotFoundError("Jalankan Layer 2 terlebih dahulu.")

PROJECT_DIR = find_project_dir()
TX_PATH = PROJECT_DIR / "outputs_ril" / "jul_transactions.csv"
RULES_PATH = PROJECT_DIR / "outputs_ril" / "apriori_rules_ril.csv"
TRAIN_PATH = PROJECT_DIR / "outputs_ril" / "layer3_train_features.csv"
TEST_PATH = PROJECT_DIR / "outputs_ril" / "layer3_test_features.csv"
TRAIN_MONTHS = ["2017-01", "2017-02"]
TEST_MONTHS = ["2017-03"]
FEATURE_COLUMNS = [
    "current_basket_size", "hour_of_day", "day_of_week", "is_weekend",
    "rule_confidence", "rule_lift", "antecedent_rate", "interest_lift",
]

print("--> [INFO] Parameter Layer 3 siap: 8 fitur konteks struk, latih=Januari-Februari, uji=Maret.")
print(f"--> [INFO] Folder proyek: {PROJECT_DIR}")


## 1. Susun konteks struk dan pasangan aturan

Ukuran keranjang dihitung dari seluruh barang di struk. Kehadiran consequent hanya masuk ke label, bukan ke fitur.


In [ ]:
print("--> [INFO] Memuat transaksi JUL dan aturan Apriori...")
tx = pd.read_csv(
    TX_PATH,
    dtype={"NO_BKT": str, "ITEM": str},
    parse_dates=["TGL_TRANS"],
)
rules = pd.read_csv(RULES_PATH, dtype={"antecedent": str, "consequent": str})
print("Aturan:", rules.shape)

hist = tx.loc[tx["bulan"].isin(TRAIN_MONTHS)]
n_hist = hist["NO_BKT"].nunique()
antecedent_rate = hist.groupby("ITEM")["NO_BKT"].nunique() / n_hist
rules["antecedent_rate"] = rules["antecedent"].map(antecedent_rate).astype("float32")
rules["rule_confidence"] = rules["confidence"].astype("float32")
rules["rule_lift"] = rules["lift"].astype("float32")
rules["interest_lift"] = rules["antecedent_rate"] * rules["rule_lift"]

context = (
    tx.groupby("NO_BKT", as_index=False)
    .agg(
        current_basket_size=("ITEM", "nunique"),
        hour_of_day=("hour_of_day", "first"),
        day_of_week=("day_of_week", "first"),
        is_weekend=("is_weekend", "first"),
        bulan=("bulan", "first"),
    )
)
presence = tx[["NO_BKT", "ITEM"]].drop_duplicates()
print("--> [INFO] Struk:", len(context), "| antecedent_rate dihitung dari", f"{n_hist:,}", "struk historis")

def build_split(months, label):
    print(f"--> [INFO] Menyusun matriks {label}...")
    basket_ids = set(context.loc[context["bulan"].isin(months), "NO_BKT"])
    present = presence.loc[presence["NO_BKT"].isin(basket_ids)]
    matched = rules.merge(
        present.rename(columns={"ITEM": "antecedent"}),
        on="antecedent",
        how="inner",
    )
    positives = present.rename(columns={"ITEM": "consequent"}).assign(y=np.int8(1))
    frame = matched.merge(positives, on=["NO_BKT", "consequent"], how="left")
    frame["y"] = frame["y"].fillna(0).astype("int8")
    frame = frame.merge(context, on="NO_BKT", how="left")
    keep = ["NO_BKT", "antecedent", "consequent", *FEATURE_COLUMNS, "y"]
    frame = frame[keep]
    for col in FEATURE_COLUMNS:
        frame[col] = frame[col].astype("float32")
    print(label, frame.shape, "proporsi kelas 1:", round(float(frame["y"].mean()), 4))
    return frame

train_df = build_split(TRAIN_MONTHS, "latih")
test_df = build_split(TEST_MONTHS, "uji")
train_df.to_csv(TRAIN_PATH, index=False)
test_df.to_csv(TEST_PATH, index=False)
print("--> [INFO] Tersimpan:", TRAIN_PATH.name, TEST_PATH.name)
